# 05 Solar Analysis

This notebook covers analysis techniques for solar images sunspot detection, limb darkening measurement, and brightness mapping.

**WARNING:** Never image the Sun without a proper solar filter serious risk to your telescope and eyes.

**Requirements:** astropy, numpy, matplotlib, scipy

In [ ]:
from astropy.io import fits
from astropy.visualization import SqrtStretch, PercentileInterval
from scipy.ndimage import label, find_objects
import numpy as np
import matplotlib.pyplot as plt

fits_file = "your_solar_image.fits"  # Change this to your file path

with fits.open(fits_file) as hdul:
    data   = hdul[0].data
    header = hdul[0].header

# Handle grayscale or color image
if data.ndim == 2:
    solar = data.copy()           # Already grayscale
else:
    solar = np.mean(data, axis=0) # Convert color to grayscale

## Displaying the Solar Disk

SqrtStretch is used for solar images it brings out faint surface detail while keeping the bright disk visible.

### Why SqrtStretch?
The square root function compresses bright values and expands faint ones good balance for the solar disk.

In [ ]:
interval = PercentileInterval(99.5)  # Ignore top 0.5% to avoid overexposure
stretch  = SqrtStretch()

vmin, vmax      = interval.get_limits(solar)
solar_stretched = stretch(np.clip((solar - vmin) / (vmax - vmin), 0, 1))

plt.figure(figsize=(10, 10))
plt.imshow(solar_stretched, cmap='hot', origin='lower')
plt.title('Solar Disk')
plt.axis('off')
plt.colorbar(label='Relative Brightness')
plt.show()

## Sunspot Detection

Sunspots are darker than the surrounding photosphere.
We find them by looking for regions below a brightness threshold.

### How it works
 `threshold` sets the cutoff regions darker than this are marked as sunspots
 `label` groups connected dark pixels into individual sunspot regions
 Adjust `0.7` up or down depending on how dark the spots appear in your image

In [ ]:
threshold    = np.mean(solar_stretched) * 0.7  # Spots darker than 70% of mean adjust as needed
sunspot_mask = solar_stretched < threshold

labeled, num_spots = label(sunspot_mask)
print(f'Sunspot regions detected: {num_spots}')

plt.figure(figsize=(10, 10))
plt.imshow(solar_stretched, cmap='hot', origin='lower')
plt.contour(sunspot_mask, levels=[0.5], colors='cyan', linewidths=1)
plt.title(f'Solar Disk {num_spots} Sunspot Regions')
plt.axis('off')
plt.show()

# Analyze each sunspot
objects = find_objects(labeled)
print('\nSunspot Analysis:')
for i, obj in enumerate(objects[:10]):  # Show first 10 spots
    region          = solar_stretched[obj]
    area            = np.sum(labeled[obj] == i + 1)
    mean_brightness = np.mean(region)
    print(f'Spot {i+1}: Area={area}px  Brightness={mean_brightness:.3f}')

## Limb Darkening

The edges of the solar disk appear darker than the center a well-known physical effect caused by the Sun's atmosphere.

### How it works
 We find the disk center (brightest point)
 Calculate distance from center for every pixel
 Plot mean brightness at each distance
 A decreasing curve confirms limb darkening